# Basic RAG with Ollama + LangChain + Chroma

A minimal, end-to-end Retrieval-Augmented Generation pipeline running fully locally:

| Piece | Choice |
|---|---|
| LLM | `llama3.2` via Ollama |
| Embeddings | `qwen3-embedding` via Ollama |
| Loaders | `WebBaseLoader`, `PyPDFLoader`, `DirectoryLoader` |
| Splitter | `RecursiveCharacterTextSplitter` |
| Vector store | Chroma (persisted to disk) |
| Retriever | Chroma similarity / MMR retriever |

**Prerequisite:** Ollama must be running (`ollama serve`) with both models pulled.

## 1. Install dependencies

In [7]:
# Run once. Restart the kernel afterwards if packages were newly installed.
%pip install -q -U langchain langchain-community langchain-ollama langchain-chroma \
    chromadb beautifulsoup4 pypdf

Note: you may need to restart the kernel to use updated packages.


## 2. Check Ollama and pull the models

In [ ]:
!ollama --version
!ollama list

In [ ]:
# Uncomment if the models are not present in `ollama list` above.
# !ollama pull llama3.2
# !ollama pull qwen3-embedding

## 3. Configuration

In [4]:
LLM_MODEL       = "llama3.2"          # generation model
EMBED_MODEL     = "qwen3-embedding"   # embedding model
OLLAMA_BASE_URL = "http://localhost:11434"
PERSIST_DIR     = "./chroma_db"       # on-disk vector store
COLLECTION_NAME = "rag_demo"

## 4. Load documents

LangChain has a loader for nearly every source. Below: a web page (default),
plus commented-out variants for PDFs, a folder of text files, and raw strings.

In [1]:
from langchain_community.document_loaders import WebBaseLoader
import bs4

loader = WebBaseLoader(
    web_paths=["https://lilianweng.github.io/posts/2023-06-23-agent/"],
    bs_kwargs={"parse_only": bs4.SoupStrainer(class_=("post-content", "post-title", "post-header"))},
)
docs = loader.load()

print(f"Loaded {len(docs)} document(s), {len(docs[0].page_content):,} chars")
print(docs[0].page_content[:400], "...")

/var/folders/qt/7qfpwbkx3cq0xb9m37811y000000gn/T/ipykernel_95051/1642298814.py:1: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import WebBaseLoader
/Users/ashishbansal/Documents/Training/Aistack_Morning/AI_Thota_Morning8AM/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
USER_AGENT environment variable not set, consider setting it to identify your requests.


Loaded 1 document(s), 43,047 chars


      LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring examples. The potentiality of LLM extends beyond generating well-written copies, stories, e ...


In [ ]:
# --- Other loaders (uncomment what you need) -------------------------------

# PDF
# from langchain_community.document_loaders import PyPDFLoader
# docs = PyPDFLoader("./my_paper.pdf").load()          # one Document per page

# A directory of .txt / .md files
# from langchain_community.document_loaders import DirectoryLoader, TextLoader
# docs = DirectoryLoader("./data", glob="**/*.md", loader_cls=TextLoader).load()

# In-memory text
# from langchain_core.documents import Document
# docs = [Document(page_content="...", metadata={"source": "notes"})]

## 5. Split into chunks

Chunks need to be small enough to retrieve precisely, big enough to stay meaningful.
`chunk_overlap` keeps sentences from being cut mid-thought at a boundary.

In [2]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200,
    add_start_index=True,   # records char offset in metadata
)
chunks = splitter.split_documents(docs)

print(f"{len(chunks)} chunks")
print(chunks[0].metadata)
print(chunks[0].page_content[:300], "...")

63 chunks
{'source': 'https://lilianweng.github.io/posts/2023-06-23-agent/', 'start_index': 8}
LLM Powered Autonomous Agents
    
Date: June 23, 2023  |  Estimated Reading Time: 31 min  |  Author: Lilian Weng


Building agents with LLM (large language model) as its core controller is a cool concept. Several proof-of-concepts demos, such as AutoGPT, GPT-Engineer and BabyAGI, serve as inspiring ...


## 6. Embed with `qwen3-embedding` and store in Chroma

The first call downloads nothing (model is local) but does run the model, so
embedding a few hundred chunks takes a moment.

In [5]:
from langchain_ollama import OllamaEmbeddings

embeddings = OllamaEmbeddings(model=EMBED_MODEL, base_url=OLLAMA_BASE_URL)

# Sanity check: what dimension does this model produce?
probe = embeddings.embed_query("hello world")
print(f"embedding dim = {len(probe)}")

embedding dim = 4096


In [8]:
from langchain_chroma import Chroma

vectorstore = Chroma.from_documents(
    documents=chunks,
    embedding=embeddings,
    collection_name=COLLECTION_NAME,
    persist_directory=PERSIST_DIR,
)
print("indexed:", vectorstore._collection.count(), "chunks")

indexed: 63 chunks


In [ ]:
# Re-opening an existing store later (skip re-embedding):
#
# vectorstore = Chroma(
#     collection_name=COLLECTION_NAME,
#     embedding_function=embeddings,
#     persist_directory=PERSIST_DIR,
# )
#
# Wiping it to start clean:
# vectorstore.delete_collection()

## 7. Build a retriever

In [ ]:
retriever = vectorstore.as_retriever(
    search_type="similarity",       # or "mmr" for diversity, "similarity_score_threshold"
    search_kwargs={"k": 4},
)

hits = retriever.invoke("What is task decomposition for LLM agents?")
for i, d in enumerate(hits, 1):
    print(f"--- [{i}] {d.metadata.get('source')} @ {d.metadata.get('start_index')}")
    print(d.page_content[:220].replace("\n", " "), "...\n")

In [ ]:
# MMR variant — trades a little relevance for less redundancy among results.
mmr_retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 4, "fetch_k": 20, "lambda_mult": 0.5},
)

## 8. Wire up the RAG chain

`retriever | format | prompt | llm | parse` — the retrieved chunks are pasted into
the prompt as context, and the model is told to answer only from them.

In [ ]:
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

llm = ChatOllama(model=LLM_MODEL, base_url=OLLAMA_BASE_URL, temperature=0)

prompt = ChatPromptTemplate.from_template(
    """You are a helpful assistant. Answer the question using ONLY the context below.
If the context does not contain the answer, say you don't know. Keep it to three sentences.

Context:
{context}

Question: {question}

Answer:"""
)

def format_docs(docs):
    return "\n\n".join(d.page_content for d in docs)

rag_chain = (
    {"context": retriever | format_docs, "question": RunnablePassthrough()}
    | prompt
    | llm
    | StrOutputParser()
)

## 9. Ask questions

In [ ]:
print(rag_chain.invoke("What is task decomposition, and what methods are used for it?"))

In [ ]:
# Streaming, token by token
for token in rag_chain.stream("What types of memory does an LLM agent have?"):
    print(token, end="", flush=True)

## 10. Return the answer *and* its sources

Useful in practice: you almost always want to show what the answer was grounded in.

In [ ]:
from langchain_core.runnables import RunnableParallel

rag_with_sources = RunnableParallel(
    {"context": retriever, "question": RunnablePassthrough()}
).assign(answer=(lambda x: {"context": format_docs(x["context"]), "question": x["question"]})
         | prompt | llm | StrOutputParser())

result = rag_with_sources.invoke("What are the main components of an LLM-powered agent?")

print(result["answer"], "\n")
print("Sources:")
for d in result["context"]:
    print(" -", d.metadata.get("source"), "@", d.metadata.get("start_index"))

## 11. Adding new documents later

Chroma is persistent, so you can incrementally grow the index without rebuilding.

In [ ]:
# new_docs = PyPDFLoader("./another.pdf").load()
# new_chunks = splitter.split_documents(new_docs)
# vectorstore.add_documents(new_chunks)
# print("now:", vectorstore._collection.count(), "chunks")

---
### Troubleshooting

- **`ConnectionError` / connection refused** — Ollama isn't running. Start it with `ollama serve`.
- **`model not found`** — `ollama pull llama3.2` / `ollama pull qwen3-embedding`.
- **Embedding step is slow** — `qwen3-embedding:latest` is the 8B variant (~4.7 GB). For a much
  faster index, pull a smaller one and set `EMBED_MODEL` accordingly, e.g.
  `ollama pull dengcao/Qwen3-Embedding-0.6B:Q8_0`.
- **Dimension mismatch on reload** — you changed `EMBED_MODEL` after building the collection.
  Delete `./chroma_db` (or `vectorstore.delete_collection()`) and re-index.
- **Poor answers** — raise `k`, tune `chunk_size`/`chunk_overlap`, or switch the retriever to `mmr`.